# Optimización de Crop3Test2

Se optimizan gamma y delta con igualdad de mezcla en cada iteración. Las pruebas usan las mismas mezclas de entrenamiento y semillas. Se maximiza SSIM y se valida sobre test con semillas independientes.


In [1]:
import importlib
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf
import optuna
import matplotlib.pyplot as plt

project_root = next(
    root for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "project" / "CROP" / "models").is_dir()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from project.loader import Loader
import project.CROP.models.crop3_test as crop3_test_module
import project.CROP.models.crop3_test2 as crop3_test2_module
importlib.reload(crop3_test_module)
importlib.reload(crop3_test2_module)

SEED = 3333
DATASET = "fashion"
CVAE_NAME = "int_256_lat_64_beta_0_4_fashion.keras"
PREDICTOR_NAME = f"CCE_Conv2D_{DATASET}.keras"
N_IMAGES = 1000
N_TRIALS = 100
N_VALIDATION_IMAGES = 1000
N_VALIDATION_RUNS = 3
TARGET_METRIC = "acc_both"  # Maximizar el SSIM medio de las estimaciones.
SEPARATION_ITERATIONS = 20
assert TARGET_METRIC in {"acc_both", "ssim", "recon_bpsnr"}

FIXED_PARAMS = {
    "alpha_1": -2,
    "alpha_2": -22,
    "bias": 0.22,
    "slope": 22,
    "beta": 1.0,
    "alpha_mix": 0.5,
    "saturation_object_threshold": 0.05,
}

tf.keras.utils.set_random_seed(SEED)
data = Loader.data(dataset=DATASET)
cvae = Loader.cvae(model_name=CVAE_NAME)
predictor = Loader.predictor(model_name=PREDICTOR_NAME)

# Generador independiente de las semillas del CVAE y de Optuna.
rng = np.random.default_rng(SEED)
idx0 = rng.permutation(len(data["x_train"]))[:N_IMAGES]
idx1 = rng.permutation(len(data["x_train_1"]))[:N_IMAGES]
train_batch = (
    data["x_train"][idx0], data["x_train_1"][idx1],
    data["y_train"][idx0], data["y_train_1"][idx1],
)


def scalar_metric(metrics, key):
    value = metrics[key]
    # Las métricas de calidad devuelven (media, desvío).
    if isinstance(value, (tuple, list)):
        value = value[0]
    return float(value)


def objective(trial):
    params = {
        **FIXED_PARAMS,
        "gamma": trial.suggest_float("gamma", 0.0, 2.0),
        "delta": trial.suggest_float("delta", 0.0, 1.0),
    }
    tf.keras.utils.set_random_seed(SEED)
    model = crop3_test2_module.Crop3Test2(cvae=cvae, predictor=predictor)
    metrics = model.unmix(
        *train_batch,
        params=params,
        iterations=SEPARATION_ITERATIONS,
        labels=data["labels"],
        show_image=False,
        show_metrics=False,
    )
    score = scalar_metric(metrics, TARGET_METRIC)
    if not np.isfinite(score):
        raise ValueError(f"Métrica no finita: {TARGET_METRIC}={score}")
    trial.set_user_attr("effective_params", params)
    for key in ("acc_both", "ssim", "recon_bpsnr"):
        trial.set_user_attr(key, scalar_metric(metrics, key))
    return score


2026-11-16 19:45:29.494862: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-11-16 19:45:29.495293: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-11-16 19:45:29.498421: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-11-16 19:45:29.508365: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1794869129.527763  387424 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1794869129.53

Usando fashion como dataset


2026-11-16 19:45:33.653885: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## Estudio

Se optimizan gamma y delta durante 20 iteraciones de separación. Cada prueba queda guardada en SQLite.


In [2]:
from datetime import datetime

output_dir = project_root / "project" / "experiments" / "optim_params" / "results"
output_dir.mkdir(parents=True, exist_ok=True)
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
study_path = output_dir / f"crop3_test2_{DATASET}_{TARGET_METRIC}_{run_id}.sqlite3"
study = optuna.create_study(
    direction="maximize",
    storage=f"sqlite:///{study_path}",
    study_name=f"Crop3Test2_{DATASET}_{TARGET_METRIC}",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
print("Estudio persistente:", study_path, flush=True)
study.optimize(objective, n_trials=N_TRIALS, n_jobs=1)
best_params = {**FIXED_PARAMS, **study.best_params}
print("Mejor valor:", study.best_value)
print("Parámetros completos:", best_params)


[I 2026-11-16 19:45:41,634] A new study created in RDB with name: Crop3Test2_fashion_acc_both


Estudio persistente: /home/galsan/Desktop/tesis/CROP_1/project/experiments/optim_params/results/crop3_test2_fashion_acc_both_20261116_194540.sqlite3


[I 2026-11-16 19:45:46,152] Trial 0 finished with value: 0.676 and parameters: {'gamma': 1.5031512144089574, 'delta': 0.10925127538854273}. Best is trial 0 with value: 0.676.
[I 2026-11-16 19:45:50,052] Trial 1 finished with value: 0.679 and parameters: {'gamma': 0.9722425581148704, 'delta': 0.4998311777480342}. Best is trial 1 with value: 0.679.
[I 2026-11-16 19:45:54,043] Trial 2 finished with value: 0.691 and parameters: {'gamma': 1.3142199980396865, 'delta': 0.2358847112408069}. Best is trial 2 with value: 0.691.
[I 2026-11-16 19:45:58,062] Trial 3 finished with value: 0.685 and parameters: {'gamma': 1.2255897813211882, 'delta': 0.11965240339743843}. Best is trial 2 with value: 0.691.
[I 2026-11-16 19:46:02,037] Trial 4 finished with value: 0.683 and parameters: {'gamma': 1.4244045932667562, 'delta': 0.37256054205032785}. Best is trial 2 with value: 0.691.
[I 2026-11-16 19:46:06,004] Trial 5 finished with value: 0.688 and parameters: {'gamma': 1.0495365456694699, 'delta': 0.2632870

KeyboardInterrupt: 

## Visualización de la optimización

Gráficos con Matplotlib: no requieren Plotly ni instalaciones dentro del notebook.

In [3]:
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
fig, ax = plt.subplots(figsize=(9, 4))
numbers = [t.number for t in completed]
values = np.array([t.value for t in completed])
ax.scatter(numbers, values, label="Prueba")
ax.plot(numbers, np.maximum.accumulate(values), label="Mejor hasta ahora", color="tab:orange")
ax.set_xlabel("Prueba")
ax.set_ylabel(TARGET_METRIC)
ax.grid(alpha=0.3)
ax.legend()
plt.show()
plt.close(fig)

names = list(study.best_params)
fig, axes = plt.subplots(1, len(names), figsize=(10, 4), squeeze=False)
for ax, name in zip(axes.flat, names):
    ax.scatter([t.params[name] for t in completed], values, alpha=0.7)
    ax.set_xlabel(name)
    ax.set_ylabel(TARGET_METRIC)
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()
plt.close(fig)


/home/galsan/Desktop/tesis/CROP_1/project/experiments/optim_params/optim_params.ipynb:cell_5:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
/home/galsan/Desktop/tesis/CROP_1/project/experiments/optim_params/optim_params.ipynb:cell_5:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Validación de Crop3Test2

Evalúa los mejores parámetros en tres mezclas de test independientes; muestra curvas e imágenes de la primera corrida.


In [4]:
best_params = {**FIXED_PARAMS, **study.best_params}
validation_results = []
validation_rng = np.random.default_rng(SEED + 1000)
for run in range(N_VALIDATION_RUNS):
    idx0 = validation_rng.permutation(len(data["x_test"]))[:N_VALIDATION_IMAGES]
    idx1 = validation_rng.permutation(len(data["x_test_1"]))[:N_VALIDATION_IMAGES]
    tf.keras.utils.set_random_seed(SEED + 1000 + run)
    validation_model = crop3_test2_module.Crop3Test2(cvae=cvae, predictor=predictor)
    metrics = validation_model.unmix(
        data["x_test"][idx0], data["x_test_1"][idx1],
        data["y_test"][idx0], data["y_test_1"][idx1],
        params=best_params.copy(),
        iterations=SEPARATION_ITERATIONS,
        labels=data["labels"],
        # Mostrar una grilla de 1000 imagenes seria ilegible. Las imagenes
        # de la primera corrida se muestran abajo con un subconjunto.
        show_image=False,
        show_metrics=False,
        show_all_curves=(run == 0),
    )
    validation_results.append({
        "run": run + 1,
        **{key: scalar_metric(metrics, key) for key in ("acc_both", "ssim", "recon_bpsnr")},
    })
    print(validation_results[-1])
    if run == 0:
        first_validation_model = validation_model
        first_validation_indices = (idx0, idx1)

# Mostrar solo 10 mezclas de la primera corrida, con sus labels finales.
from project.helper.graphics.graphics import Graphics
m = first_validation_model
idx0, idx1 = first_validation_indices
limit = min(10, len(idx0), len(idx1))
Graphics.complete_plot(
    tf.convert_to_tensor(m.mixed_input[:limit]),
    tf.convert_to_tensor(data["x_test"][idx0[:limit]]),
    tf.convert_to_tensor(data["x_test_1"][idx1[:limit]]),
    data["y_test"][idx0[:limit]], data["y_test_1"][idx1[:limit]],
    tf.convert_to_tensor(m.source1_estimation[:limit]),
    tf.convert_to_tensor(m.source2_estimation[:limit]),
    tf.convert_to_tensor(m.mask1[:limit]), tf.convert_to_tensor(m.mask2[:limit]),
    m.predictions1[:limit], m.predictions2[:limit],
    tf.convert_to_tensor(m.source1_estimation[:limit]),
    model_params=best_params,
    class_labels=data["labels"],
)


/home/galsan/Desktop/tesis/CROP_1/project/helper/graphics/graphics.py:312: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
{'run': 1, 'acc_both': 0.631, 'ssim': 0.719373881816864, 'recon_bpsnr': 19.833528518676758}
{'run': 2, 'acc_both': 0.664, 'ssim': 0.7168042063713074, 'recon_bpsnr': 19.740947723388672}
{'run': 3, 'acc_both': 0.634, 'ssim': 0.7167844176292419, 'recon_bpsnr': 19.848278045654297}
/home/galsan/Desktop/tesis/CROP_1/project/helper/graphics/graphics.py:232: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Resultados de validación

In [5]:
validation_summary = {}
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, key in zip(axes, ("acc_both", "ssim", "recon_bpsnr")):
    values = np.array([row[key] for row in validation_results])
    validation_summary[key] = {"mean": float(values.mean()), "std": float(values.std())}
    ax.plot(np.arange(1, len(values) + 1), values, "o-")
    ax.axhline(values.mean(), color="tab:orange", linestyle="--")
    ax.set_title(f"{key}: {values.mean():.4f} ± {values.std():.4f}")
    ax.set_xlabel("Corrida de validación")
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()
plt.close(fig)
print(validation_summary)


/home/galsan/Desktop/tesis/CROP_1/project/experiments/optim_params/optim_params.ipynb:cell_9:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
{'acc_both': {'mean': 0.6429999999999999, 'std': 0.014899664425751354}, 'ssim': {'mean': 0.7176541686058044, 'std': 0.0012160477087126}, 'recon_bpsnr': {'mean': 19.807584762573242, 'std': 0.04750269030963585}}


## Guardar estudio y resultados

Guarda parámetros completos, todas las pruebas y validación en un archivo pickle; no guarda batches de imágenes.

In [6]:
import pickle
from datetime import datetime

output_dir = project_root / "project" / "experiments" / "optim_params" / "results"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"crop3_test2_{DATASET}_{TARGET_METRIC}_{datetime.now():%Y%m%d_%H%M%S}.pkl"
with output_path.open("wb") as stream:
    pickle.dump({
        "model": "Crop3Test2",
        "dataset": DATASET,
        "cvae": CVAE_NAME,
        "predictor": PREDICTOR_NAME,
        "seed": SEED,
        "objective": TARGET_METRIC,
        "best_params": best_params,
        "study": study,
        "validation_results": validation_results,
        "validation_summary": validation_summary,
    }, stream)
print("Guardado:", output_path)


Guardado: /home/galsan/Desktop/tesis/CROP_1/project/experiments/optim_params/results/crop3_test2_fashion_ssim_20261116_181256.pkl
